# Revenue Regression — Mail-Order Pharmacy

**Business goal.** The pharmacy uses daily dynamic pricing to maximise revenue. For
each ordered line item we want to predict the **revenue** generated, given product
attributes, price, competitor price and behavioural / temporal context.

**Performance measures.** RMSE, MAE and R² on a chronologically held-out test set,
plus per-day diagnostics.

**Approach (CRISP-DM).**
1. *Data understanding* — load and merge `train.csv` with `items.csv`.
2. *Data preparation* — chronological split first, **then** all feature engineering
   on the train portion only to prevent leakage.
3. *Feature engineering* — per-pid behavioural rates, smoothed target encodings,
   competitor-price dynamics, days-since-promo, leak-safe product age.
4. *Modeling* — three tree-based regressors (Decision Tree / Random Forest /
   XGBoost), tuned with Optuna using **day-aware** cross-validation on the
   training set.
5. *Evaluation* — RMSE / MAE / R² on the test set and **per-day** diagnostics.

In [ ]:
# Imports & configuration
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from scipy.sparse            import hstack, csr_matrix
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
OUT_DIR = '../data/processed/modeling/regression'
os.makedirs(OUT_DIR, exist_ok=True)

# HPO budget — tuned for strong XGBoost performance inside a 2 h budget on 16 GB.
# XGB gets a 200k sub-sample (fast on hist tree_method); RF runs on 100k because
# sklearn's RF on sparse data is much slower per fit.
HPO_SUBSAMPLE_XGB  = 200_000     # rows for XGBoost HPO
HPO_SUBSAMPLE_RF   = 100_000     # rows for Decision Tree / Random Forest HPO
N_TRIALS_DT        = 20
N_TRIALS_RF        = 12
N_TRIALS_XGB       = 50          # most trials go to XGB — usually the strongest learner
N_HPO_FOLDS        = 3
CORR_DROP_THRESH   = 0.95        # |corr| above which one of the pair is dropped

## 1. Data understanding — load and merge

`train.csv` records every user action (click / basket / order) with daily price and
competitor price. `items.csv` holds the static product attributes. Joined on `pid`.

In [ ]:
train = pd.read_csv('../data/raw/train.csv', sep='|')
items = pd.read_csv('../data/raw/items.csv', sep='|')
df = train.merge(items, on='pid', how='left', validate='m:1')

# Downcast to save memory
for col in ('day', 'pid', 'adFlag', 'availability', 'click', 'basket', 'order',
            'manufacturer', 'salesIndex', 'genericProduct', 'category'):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
for col in ('price', 'rrp', 'competitorPrice', 'revenue'):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

del train, items
gc.collect()
print(f"Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory:         {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Chronological train / test split

The dataset spans 92 days. We pick the day that puts ≈ 80 % of rows in train and
20 % in test. **All subsequent feature engineering is done on the train portion
only**, then applied to the test portion — that is the only safe order of operations
under time-series data.

In [ ]:
df = df.sort_values(['day', 'lineID']).reset_index(drop=True)

unique_days = np.sort(df['day'].unique())
cumulative_rows = df.groupby('day').size().cumsum()
split_target = int(len(df) * 0.8)
TRAIN_CUTOFF_DAY = int(cumulative_rows[cumulative_rows >= split_target].index[0])

train_full = df[df['day'] <= TRAIN_CUTOFF_DAY].copy()
test_full  = df[df['day'] >  TRAIN_CUTOFF_DAY].copy()

del df
gc.collect()

print(f"Train cutoff day: {TRAIN_CUTOFF_DAY}")
print(f"Train rows (full, all events): {len(train_full):,}  "
      f"days {train_full['day'].min()}–{train_full['day'].max()}")
print(f"Test  rows (full, all events): {len(test_full):,}  "
      f"days {test_full['day'].min()}–{test_full['day'].max()}")

## 3. Leak-safe feature engineering

Everything below is computed **without using future or test information**:

* **Per-pid behavioural rates** — `pid_order_rate`, `pid_click_rate`,
  `pid_basket_rate`, `pid_price_std`, `pid_seen_in_train`. Means/stds taken over the
  train period; for products absent in train (cold start) we fall back to the global
  mean.
* **Smoothed target encodings** for `manufacturer`, `group`, `category` —
  smoothing prevents over-fitting to rare categories, the encoding target is
  `log1p(revenue)` evaluated on train orders only.
* **Competitor-price dynamics** — 7-day rolling mean and trend (slope of last 7
  observations per pid). `shift(1)` excludes the current row.
* **Days since last promo** — per pid, days since the most recent `adFlag == 1` row
  observed *strictly before* the current one (999 if no prior promo).
* **Leak-safe product age** — `day − first_seen_in_train`, with `first_seen_in_train`
  taken from the train partition only.

In [ ]:
# Per-pid behavioural rates (computed once from train_full)
TE_SMOOTHING = 20

global_rev_log   = np.log1p(train_full.loc[train_full['order'] == 1, 'revenue']).mean()
global_order_rt  = train_full['order'].mean()
global_click_rt  = train_full['click'].mean()
global_basket_rt = train_full['basket'].mean()

pid_stats = train_full.groupby('pid').agg(
    pid_order_rate  = ('order',  'mean'),
    pid_click_rate  = ('click',  'mean'),
    pid_basket_rate = ('basket', 'mean'),
    pid_price_std   = ('price',  'std'),
).reset_index()
pid_stats['pid_price_std']     = pid_stats['pid_price_std'].fillna(0.0)
pid_stats['pid_seen_in_train'] = 1

def _attach_pid_stats(d):
    d = d.merge(pid_stats, on='pid', how='left')
    d['pid_order_rate']    = d['pid_order_rate'].fillna(global_order_rt)
    d['pid_click_rate']    = d['pid_click_rate'].fillna(global_click_rt)
    d['pid_basket_rate']   = d['pid_basket_rate'].fillna(global_basket_rt)
    d['pid_price_std']     = d['pid_price_std'].fillna(0.0)
    d['pid_seen_in_train'] = d['pid_seen_in_train'].fillna(0).astype('int8')
    return d

train_full = _attach_pid_stats(train_full)
test_full  = _attach_pid_stats(test_full)

# Smoothed target encodings — fit on train orders only
def _smoothed_te(train_orders, key, target_col='log1p_rev'):
    g          = train_orders.groupby(key)[target_col]
    counts     = g.count()
    means      = g.mean()
    smoothed   = (counts * means + TE_SMOOTHING * global_rev_log) / (counts + TE_SMOOTHING)
    return smoothed

train_orders_mask = train_full['order'] == 1
train_orders_for_te = train_full.loc[train_orders_mask, ['manufacturer', 'group', 'category']].copy()
train_orders_for_te['log1p_rev'] = np.log1p(train_full.loc[train_orders_mask, 'revenue'].values)

te_maps = {
    'manufacturer': _smoothed_te(train_orders_for_te, 'manufacturer'),
    'group':        _smoothed_te(train_orders_for_te, 'group'),
    'category':     _smoothed_te(train_orders_for_te, 'category'),
}
del train_orders_for_te

for col, mapping in te_maps.items():
    train_full[f'{col}_te'] = train_full[col].map(mapping).fillna(global_rev_log).astype('float32')
    test_full[f'{col}_te']  = test_full[col].map(mapping).fillna(global_rev_log).astype('float32')

print("Per-pid + target encoding features added.")
print(f"  Unique pids in train: {len(pid_stats):,}")
print(f"  TE coverage (test):  manufacturer {test_full['manufacturer'].isin(te_maps['manufacturer'].index).mean():.3f}, "
      f"group {test_full['group'].isin(te_maps['group'].index).mean():.3f}")

In [ ]:
# Competitor-price dynamics + days-since-promo + base price features
def _add_dynamics_and_promo(d):
    d = d.sort_values(['pid', 'day']).reset_index(drop=True)
    pid_series = d['pid']

    # Shift to exclude today; group-aware rolling so windows never cross pids
    grp = d.groupby('pid', sort=False)['competitorPrice']
    shifted_cp = grp.shift(1)

    d['competitorPrice_7day_avg'] = (
        shifted_cp.groupby(pid_series)
                  .rolling(7, min_periods=1).mean()
                  .reset_index(level=0, drop=True)
                  .astype('float32')
    )

    # 7-day trend: (shift(1) − shift(7)) / 6  — vectorised approximation to the
    # OLS slope on the last seven observations. Much faster than rolling.apply
    # while preserving the up-/down-trend signal.
    shifted_cp_7 = grp.shift(7)
    d['competitorPrice_trend'] = (
        ((shifted_cp - shifted_cp_7) / 6.0).fillna(0.0).astype('float32')
    )

    # Days since last promo (adFlag == 1, strictly past) — per pid
    last_promo_day = pd.Series(
        np.where(d['adFlag'] == 1, d['day'], np.nan),
        index=d.index,
    )
    last_promo_day = (last_promo_day.groupby(pid_series).shift(1)
                                    .groupby(pid_series).ffill())
    d['days_since_promo'] = ((d['day'] - last_promo_day)
                             .fillna(999).astype('int16'))
    return d

train_full = _add_dynamics_and_promo(train_full)
test_full  = _add_dynamics_and_promo(test_full)

# Base price features
for d in (train_full, test_full):
    d['priceRatio']             = (d['price'] / d['rrp'].replace(0, np.nan)).fillna(1.0).astype('float32')
    d['priceVsCompetitor']      = (d['price'] / d['competitorPrice'].replace(0, np.nan)).fillna(1.0).astype('float32')
    d['priceDiscount']          = (d['rrp'] - d['price']).clip(lower=0).astype('float32')
    d['missingCompetitorPrice'] = d['competitorPrice'].isnull().astype('int8')
    wd = (d['day'] % 7).replace({0: 7})
    d['weekDay_sin'] = np.sin(2 * np.pi * wd / 7).astype('float32')
    d['weekDay_cos'] = np.cos(2 * np.pi * wd / 7).astype('float32')
    d['adFlag_x_priceRatio'] = (d['adFlag'] * d['priceRatio']).astype('float32')
    d['regulated_generic']   = (d['salesIndex'] * d['genericProduct'].fillna(0)).astype('float32')

# Leak-safe product age — first_seen taken from train_full only
first_seen_map = train_full.groupby('pid')['day'].min().astype('int16')
for d in (train_full, test_full):
    d['product_age_days'] = (d['day'] - d['pid'].map(first_seen_map).fillna(d['day'])).astype('int16')

print("Dynamics, promo, base + age features added.")
print(f"  Train memory: {train_full.memory_usage(deep=True).sum() / 1e6:.0f} MB  | "
      f"Test memory: {test_full.memory_usage(deep=True).sum() / 1e6:.0f} MB")

## 4. Filter to orders and drop highly correlated features

Revenue is only realised on order rows, so the regression target is defined on
`order == 1`. After filtering we look at the correlation matrix of the numeric
features and drop one of each pair with `|corr| > 0.95` to reduce multicollinearity
and feature count (keeps the linear model honest and shortens RF/XGB fits).

In [ ]:
train_orders = train_full[train_full['order'] == 1].copy()
test_orders  = test_full[test_full['order'] == 1].copy()

# Free the larger frames before continuing
del train_full, test_full
gc.collect()

print(f"Train orders: {len(train_orders):,}  |  Test orders: {len(test_orders):,}")

NUM_FEATURES_RAW = [
    'price', 'rrp', 'competitorPrice',
    'priceRatio', 'priceVsCompetitor', 'priceDiscount', 'missingCompetitorPrice',
    'weekDay_sin', 'weekDay_cos',
    'adFlag_x_priceRatio', 'regulated_generic',
    'pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
    'pid_price_std', 'pid_seen_in_train',
    'manufacturer_te', 'group_te', 'category_te',
    'competitorPrice_7day_avg', 'competitorPrice_trend',
    'days_since_promo', 'product_age_days',
]
CAT_FEATURES = ['adFlag', 'availability', 'pharmForm',
                'genericProduct', 'salesIndex', 'campaignIndex']

# Correlation-based drop
corr_mat = train_orders[NUM_FEATURES_RAW].fillna(0).corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape, dtype=bool), k=1))
DROPPED_HIGH_CORR = [c for c in upper.columns if any(upper[c] > CORR_DROP_THRESH)]
NUM_FEATURES = [c for c in NUM_FEATURES_RAW if c not in DROPPED_HIGH_CORR]

print(f"Dropped {len(DROPPED_HIGH_CORR)} highly correlated features: {DROPPED_HIGH_CORR}")
print(f"Numeric features kept: {len(NUM_FEATURES)},  Categorical: {len(CAT_FEATURES)}")

## 5. Preprocessing pipeline

* Median imputation + `RobustScaler` on numeric features (robust to revenue outliers).
* One-hot encoding on categoricals with `min_frequency=20` to cap cardinality.
* Target is `log1p(revenue)`; predictions are clipped to the training range and
  `expm1`-transformed back to euros before metrics are computed.

In [ ]:
y_train_raw = train_orders['revenue'].values.astype(np.float32)
y_test_raw  = test_orders['revenue'].values.astype(np.float32)
y_train_log = np.log1p(y_train_raw)
y_test_log  = np.log1p(y_test_raw)

train_days = train_orders['day'].values.astype(np.int16)
test_days  = test_orders['day'].values.astype(np.int16)

X_train = train_orders[NUM_FEATURES + CAT_FEATURES].copy()
X_test  = test_orders[NUM_FEATURES + CAT_FEATURES].copy()

# Categoricals: fill NA + cast to string for OneHot
for d in (X_train, X_test):
    d.fillna({c: 'Missing' for c in CAT_FEATURES}, inplace=True)
    d[CAT_FEATURES] = d[CAT_FEATURES].astype(str)

imputer = SimpleImputer(strategy='median')
scaler  = RobustScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, min_frequency=20)

X_train_num = scaler.fit_transform(imputer.fit_transform(X_train[NUM_FEATURES]))
X_test_num  = scaler.transform(imputer.transform(X_test[NUM_FEATURES]))

X_train_cat = encoder.fit_transform(X_train[CAT_FEATURES])
X_test_cat  = encoder.transform(X_test[CAT_FEATURES])

X_train_proc = hstack([csr_matrix(X_train_num.astype(np.float32)), X_train_cat]).tocsr()
X_test_proc  = hstack([csr_matrix(X_test_num.astype(np.float32)),  X_test_cat]).tocsr()

del X_train, X_test, train_orders, test_orders
gc.collect()

print(f"Processed shape — train {X_train_proc.shape}, test {X_test_proc.shape}")
print(f"Train sparsity: {1 - X_train_proc.nnz / np.prod(X_train_proc.shape):.3f}")

## 6. Day-aware cross-validation folds

Standard `KFold` would leak future days into training during CV. We instead build
**expanding-window day-aware folds**: each fold's training partition contains only
days strictly earlier than its validation partition.

In [ ]:
def make_day_folds(day_array, n_splits=3):
    """Expanding-window folds on integer day labels."""
    unique = np.sort(np.unique(day_array))
    fold_size = max(1, len(unique) // (n_splits + 1))
    folds = []
    for i in range(n_splits):
        train_end = (i + 1) * fold_size
        val_end   = min(train_end + fold_size, len(unique))
        train_days_fold = set(unique[:train_end])
        val_days_fold   = set(unique[train_end:val_end])
        tr = np.where(np.isin(day_array, list(train_days_fold)))[0]
        va = np.where(np.isin(day_array, list(val_days_fold)))[0]
        if len(tr) > 0 and len(va) > 0:
            folds.append((tr, va))
    return folds

# Build two HPO subsamples for tractable Optuna runs:
#   * RF/DT — sklearn is slow on sparse, so use a smaller sample.
#   * XGB   — hist tree method is fast, can use a larger sample.
rng = np.random.default_rng(RANDOM_STATE)

def _subsample(n_target):
    idx = rng.choice(X_train_proc.shape[0],
                     size=min(n_target, X_train_proc.shape[0]),
                     replace=False)
    return np.sort(idx)

idx_rf  = _subsample(HPO_SUBSAMPLE_RF)
idx_xgb = _subsample(HPO_SUBSAMPLE_XGB)

X_hpo_rf,  y_hpo_rf  = X_train_proc[idx_rf],  y_train_log[idx_rf]
X_hpo_xgb, y_hpo_xgb = X_train_proc[idx_xgb], y_train_log[idx_xgb]

hpo_folds_rf  = make_day_folds(train_days[idx_rf],  n_splits=N_HPO_FOLDS)
hpo_folds_xgb = make_day_folds(train_days[idx_xgb], n_splits=N_HPO_FOLDS)

print(f"HPO subsample (RF/DT): {X_hpo_rf.shape[0]:,} rows  |  "
      f"fold sizes: {[len(va) for _, va in hpo_folds_rf]}")
print(f"HPO subsample (XGB):   {X_hpo_xgb.shape[0]:,} rows  |  "
      f"fold sizes: {[len(va) for _, va in hpo_folds_xgb]}")

## 7. Optuna hyperparameter optimisation

All three tree-based models are tuned with Optuna on the day-aware CV folds,
scoring `neg_root_mean_squared_error` on the log-target (so the back-transform
never runs during CV).

* **Decision Tree** — single regression tree, our interpretable baseline.
* **Random Forest** — bagged ensemble of decision trees.
* **XGBoost** — gradient-boosted decision trees (state-of-the-art on tabular).

In [ ]:
def objective_dt(trial):
    params = {
        'max_depth':         trial.suggest_int('max_depth', 5, 22),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 10, 200),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
    }
    model = DecisionTreeRegressor(**params, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_hpo_rf, y_hpo_rf, cv=hpo_folds_rf,
                             scoring='neg_root_mean_squared_error', n_jobs=1)
    return scores.mean()

def objective_rf(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 80, 200),
        'max_depth':        trial.suggest_int('max_depth', 8, 22),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 200),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
    }
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=RANDOM_STATE)
    scores = cross_val_score(model, X_hpo_rf, y_hpo_rf, cv=hpo_folds_rf,
                             scoring='neg_root_mean_squared_error', n_jobs=1)
    return scores.mean()                       # less-negative = better

def objective_xgb(trial):
    # Wider, slower-learning-rate / more-trees space — usually the strongest setup
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 500, 2500),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.10, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel':trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 30),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
    }
    model = xgb.XGBRegressor(**params, tree_method='hist',
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    scores = cross_val_score(model, X_hpo_xgb, y_hpo_xgb, cv=hpo_folds_xgb,
                             scoring='neg_root_mean_squared_error', n_jobs=1)
    return scores.mean()

gc.collect()

print(f"Optuna HPO — Decision Tree ({N_TRIALS_DT} trials) ...")
study_dt = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_dt.optimize(objective_dt, n_trials=N_TRIALS_DT, show_progress_bar=False)
print(f"  Best CV log-RMSE: {-study_dt.best_value:.4f}")
print(f"  Best params:      {study_dt.best_params}")

print(f"\nOptuna HPO — Random Forest ({N_TRIALS_RF} trials) ...")
study_rf = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=N_TRIALS_RF, show_progress_bar=False)
print(f"  Best CV log-RMSE: {-study_rf.best_value:.4f}")
print(f"  Best params:      {study_rf.best_params}")

print(f"\nOptuna HPO — XGBoost ({N_TRIALS_XGB} trials) ...")
study_xgb = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB, show_progress_bar=False)
print(f"  Best CV log-RMSE: {-study_xgb.best_value:.4f}")
print(f"  Best params:      {study_xgb.best_params}")

del X_hpo_rf, y_hpo_rf, X_hpo_xgb, y_hpo_xgb
gc.collect()

## 8. Final training on full train set & evaluation

Each tuned model is refit on the **full** training data and evaluated on the
test split. RMSE / MAE / R² are computed on the real-euro scale.

In [ ]:
LOG_CLIP_MAX = float(np.log1p(y_train_raw.max()) + 0.5)

def fit_and_score(name, model):
    model.fit(X_train_proc, y_train_log)
    pred_log = np.clip(model.predict(X_test_proc), 0.0, LOG_CLIP_MAX)
    pred     = np.expm1(pred_log)
    rmse = np.sqrt(mean_squared_error(y_test_raw, pred))
    mae  = mean_absolute_error(y_test_raw, pred)
    r2   = r2_score(y_test_raw, pred)
    print(f"{name:24s}  RMSE = {rmse:6.3f}   MAE = {mae:6.3f}   R² = {r2:.4f}")
    return {'name': name, 'model': model, 'pred': pred,
            'rmse': rmse, 'mae': mae, 'r2': r2}

results = {}
print("Fitting final models on full train set ...\n")

results['Decision Tree'] = fit_and_score(
    'Decision Tree',
    DecisionTreeRegressor(**study_dt.best_params, random_state=RANDOM_STATE),
)

results['Random Forest'] = fit_and_score(
    'Random Forest',
    RandomForestRegressor(**study_rf.best_params, n_jobs=-1, random_state=RANDOM_STATE),
)

results['XGBoost'] = fit_and_score(
    'XGBoost',
    xgb.XGBRegressor(**study_xgb.best_params, tree_method='hist',
                     random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
)

## 9. Results table

In [ ]:
results_df = (pd.DataFrame([{'model': r['name'], 'rmse': r['rmse'],
                              'mae': r['mae'], 'r2': r['r2']}
                             for r in results.values()])
                 .sort_values('rmse')
                 .reset_index(drop=True))

results_df.to_csv(f'{OUT_DIR}/regression_results.csv', index=False)
print("Test-set performance (real-euro revenue):")
print(results_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

## 10. Per-day diagnostics

For each model we compute RMSE, MAE and R² on each individual test day. This
exposes whether errors concentrate on specific days (e.g. promo days, weekends).

In [ ]:
per_day_rows = []
for name, r in results.items():
    pred = r['pred']
    dfd = pd.DataFrame({'day': test_days, 'y': y_test_raw, 'p': pred})
    for day, grp in dfd.groupby('day'):
        if len(grp) < 30:
            continue
        per_day_rows.append({
            'model': name,
            'day':   int(day),
            'n':     len(grp),
            'rmse':  float(np.sqrt(mean_squared_error(grp['y'], grp['p']))),
            'mae':   float(mean_absolute_error(grp['y'], grp['p'])),
            'r2':    float(r2_score(grp['y'], grp['p'])) if grp['y'].nunique() > 1 else np.nan,
        })

per_day = pd.DataFrame(per_day_rows)
per_day.to_csv(f'{OUT_DIR}/regression_per_day_metrics.csv', index=False)

print("Per-day test metrics (mean ± std across days):")
print(per_day.groupby('model')[['rmse', 'mae', 'r2']]
              .agg(['mean', 'std']).round(4))

In [ ]:
# Per-day RMSE plot
fig, ax = plt.subplots(figsize=(11, 4.5))
for name in results:
    sub = per_day[per_day['model'] == name].sort_values('day')
    ax.plot(sub['day'], sub['rmse'], marker='o', label=name, linewidth=1.5)
ax.set_xlabel('Day')
ax.set_ylabel('RMSE (€)')
ax.set_title('Per-day test RMSE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/regression_per_day_rmse.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Diagnostics — best model

In [ ]:
best_name = results_df.iloc[0]['model']
best_pred = results[best_name]['pred']
print(f"Best model on RMSE: {best_name}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cap = np.percentile(y_test_raw, 99)
mask = y_test_raw <= cap
axes[0].scatter(y_test_raw[mask], best_pred[mask], alpha=0.25, s=8, color='#4C72B0')
axes[0].plot([0, cap], [0, cap], 'r--', linewidth=1.5, label='perfect prediction')
axes[0].set_xlabel('Actual revenue (€)')
axes[0].set_ylabel('Predicted revenue (€)')
axes[0].set_title(f'Actual vs predicted — {best_name}')
axes[0].legend()

residuals = y_test_raw - best_pred
clip = np.percentile(np.abs(residuals), 99)
axes[1].hist(residuals[np.abs(residuals) < clip], bins=60,
             color='#DD8452', edgecolor='white', alpha=0.9)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (actual − predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residuals — {best_name}')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/regression_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Feature importance — Random Forest

In [ ]:
feature_names = NUM_FEATURES + list(encoder.get_feature_names_out(CAT_FEATURES))
importances = pd.Series(results['Random Forest']['model'].feature_importances_,
                        index=feature_names)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index[::-1], top15.values[::-1], color='#4C72B0')
ax.set_xlabel('Importance')
ax.set_title('Top 15 features — Random Forest')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/regression_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

top15.to_csv(f'{OUT_DIR}/regression_feature_importance.csv', header=['importance'])

## 13. Model comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, metric, title, color in zip(
    axes,
    ['rmse', 'mae', 'r2'],
    ['RMSE (lower is better)', 'MAE (lower is better)', 'R² (higher is better)'],
    ['#4C72B0', '#DD8452', '#55A868'],
):
    ax.barh(results_df['model'], results_df[metric], color=color, edgecolor='white')
    ax.set_xlabel(metric.upper())
    ax.set_title(title)

plt.suptitle('Regression model comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/regression_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nAll outputs saved to {OUT_DIR}")